<h2>🧪 Quick Sanity Check</h2>

<div style="padding: 10px 12px; border-left: 6px solid #10B981; background: #ECFDF5; border-radius: 10px;">
Optional mini-cell to confirm the kernel is alive ✅
</div>

<h1>⚙️ Setup</h1>

<div style="padding: 10px 12px; border-left: 6px solid #F59E0B; background: #d6b941ff; border-radius: 10px;">
Install + upgrade key libraries (UnsLoTh + TRL + Transformers).<br>
If you’re on Colab/Kaggle, this cell handles the heavy lifting 🧰
</div>

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):    
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
# !pip install -U bitsandbytes

In [3]:
import unsloth 

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


<h1 style="color: #3B82F6;">🤖 Model</h1>

<h2 style="color: #3B82F6;">📥 Load Base Model</h2>

<div style="padding: 10px 12px; border-left: 6px solid #3B82F6; background: #EFF6FF; border-radius: 10px;">
Loads <b>unsloth/gpt-oss-20b</b> in 4-bit for VRAM-friendly training 🧊
</div>

In [ ]:
# import kagglehub 
# path = kagglehub.model_download("barnobarno/gpt-oss-20b/transformers/unsloth")

# print("Path to model files:", path)

Path to model files: /root/.cache/kagglehub/models/barnobarno/gpt-oss-20b/transformers/unsloth/1


In [4]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048   # Reduced from 2048 - speeds up generation significantly
 # Larger rank = smarter, but slower
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    max_seq_length = max_seq_length,
    local_files_only =  True
   # offload_embedding = True, # Reduces VRAM by 1GB
)

==((====))==  Unsloth 2026.1.2: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


AttributeError: 'NoneType' object has no attribute 'endswith'

In [ ]:
lora_rank = 8

<h2 style="color: #265ccfff;">🧩 LoRA Settings</h2>

<div style="padding: 10px 12px; border-left: 6px solid #8B5CF6; background: #654ed5ff; border-radius: 10px;">
Pick a LoRA rank. Higher = more capacity, slower + more memory 🔧
</div>

<h1 style="color: #1757ebff;">📚 Data</h1>

<h2>🧾 Load OlymMATH (en-hard)</h2>

<div style="padding: 10px 12px; border-left: 6px solid #06B6D4; background: #ECFEFF; border-radius: 10px;">
Pulls the dataset + previews it in a DataFrame 🗂️
</div>

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("RUC-AIBOX/OlymMATH", "en-hard")

In [ ]:
import pandas as pd 
data = pd.DataFrame(ds['test'])
print(data.head())


<h1 style="color: #b0de4eff;">🧠 Prompting</h1>

<h2 style="color: #3e1492ff;">📝 System + User Prompt Template</h2>

<div style="padding: 10px 12px; border-left: 6px solid #22C55E; background: #F0FDF4; border-radius: 10px;">
Defines the instruction style: step-by-step reasoning + final integer inside <b>\\boxed{}</b> ✅
</div>

In [ ]:
SYSTEM_PROMPT = """You are a math problem solver. Solve the problem step by step.
Finally, return only the final answer as an integer inside \\boxed{}."""

def format_prompt(question):
    return [
        {"role": "user", "content": f"{question}\n\nPlease reason step by step and put your final integer answer in \\boxed{{}}."}
    ]

<h2 style="color: #e323a9ff;">🔌 LoRA: Add Adapters</h2>

<div style="padding: 10px 12px; border-left: 6px solid #8B5CF6; background: #F5F3FF; border-radius: 10px;">
Attaches LoRA modules to attention + MLP layers (fast, memory-friendly fine-tuning) 🧬
</div>

In [ ]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA adapters added successfully!")

<h2 style="color: #e323a9ff;">🧪 Build a Tiny Training Set</h2>

<div style="padding: 10px 12px; border-left: 6px solid #06B6D4; background: #ECFEFF; border-radius: 10px;">
Creates a small list of prompt/answer pairs for quick testing 🧫
</div>

In [ ]:
# Prepare dataset - using last 50 problems for testing
import re

# Create training dataset from last 50 problems
train_data = []
for i in range(len(data)-50, len(data)):
    row = data.iloc[i]
    question = row['problem']
    answer = row['answer']  # The ground truth answer
    
    train_data.append({
        "prompt": format_prompt(question),
        "answer": str(answer),
        "reasoning_effort": "low"
    })

print(f"Created dataset with {len(train_data)} problems")
print(f"Sample question: {train_data[0]['prompt'][0]['content'][:200]}...")
print(f"Sample answer: {train_data[0]['answer']}")

<h1 style="color: #e323a9ff;">🧰 Parsing Utilities</h1>

<h2 style="color: #ffffffff;">📦 NO NEED TO CHANGE <span style="color:#4F46E5;"><b></b></span></h2>

<div style="padding: 10px 12px; border-left: 6px solid #4F46E5; background: #EEF2FF; border-radius: 10px;">
We score outputs by pulling the final value from <b>\\boxed{...}</b> 🔍
</div>

In [ ]:
"""LOOKS OK HERE"""




# Answer extraction function
def extract_boxed_answer(text):
    """Extract the answer from \\boxed{} in the model output"""
    # Try to find \boxed{...}
    patterns = [
        r'\\boxed\{([^{}]*)\}',  # Simple \boxed{answer}
        r'\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}',  # Nested braces
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text)
        if matches:
            # Return the last match (final answer)
            answer_str = matches[-1].strip()
            try:
                # Try to parse as number
                # Handle fractions, negatives, etc.
                answer_str = answer_str.replace(',', '')  # Remove commas
                if '/' in answer_str:
                    # Handle fractions
                    parts = answer_str.split('/')
                    return float(parts[0]) / float(parts[1])
                return float(answer_str)
            except:
                return None
    return None
"""The above function looks ok , keep as is"""



def parse_true_answer(answer_str):
    """Parse the ground truth answer to a number"""
    try:
        answer_str = str(answer_str).strip().replace(',', '')
        if '/' in answer_str:
            parts = answer_str.split('/')
            return float(parts[0]) / float(parts[1])
        return float(answer_str)
    except:
        return None

# Test extraction
test_text = "The answer is \\boxed{42}"
print(f"Test extraction: {extract_boxed_answer(test_text)}")

<h1 style="color: #e323a9ff;">🎯 Rewards</h1>

<h2 style="color: #ffffffff;">✨ I SHOULD ADD A HARMONY REWARD + A TOKEN USAGE REWARD</h2>

<div style="padding: 10px 12px; border-left: 6px solid #F97316; background: #d392b3ff; border-radius: 10px;">
Two rewards:
<ul style="margin: 6px 0 0 18px;">
  <li><b>Format</b> reward for including <b>\\boxed{}</b> 🧾</li>
  <li><b>Answer</b> reward based on distance from the true value 🎯</li>
</ul>
</div>

In [ ]:
"""THIS FUNCTION NEEDS TO BE TURNED INTO REWARD FUNCTION"""

# def repair_harmony_tokens(self, token_ids: list[int]) -> list[int]:
#         """
#         Robustly repairs malformed Harmony messages where <|message|> is skipped.
#         Specifically handles 'commentary' split into two tokens.
#         """
#         CHANNEL_TOKEN = 200005
#         MESSAGE_TOKEN = 200008
        
#         new_ids = []
#         i = 0
#         n = len(token_ids)
        
#         while i < n:
#             token = token_ids[i]
#             new_ids.append(token)
            
#             # Trigger only on <|channel|>
#             if token == CHANNEL_TOKEN:
#                 # Default channel length is 1 token
#                 channel_len = 1
                
#                 # Check for special case: 'commentary' split into two tokens
#                 if i + 2 < n:
#                     try:
#                         # Decode next two tokens
#                         chunk = token_ids[i+1 : i+3]
#                         decoded_chunk = self.tokenizer.decode(chunk).lower()
#                         if "commentary" in decoded_chunk:
#                             channel_len = 2
#                     except Exception:
#                         pass
                
#                 # Copy the channel tokens
#                 for _ in range(channel_len):
#                     if i + 1 < n:
#                         i += 1
#                         new_ids.append(token_ids[i])
                
#                 # Check if the NEXT token is <|message|>. If not, inject it.
#                 if i + 1 < n:
#                      if token_ids[i+1] != MESSAGE_TOKEN:
#                         new_ids.append(MESSAGE_TOKEN)
#                 else: 
#                      # End of sequence, better close with message token so parser doesn't fail
#                      new_ids.append(MESSAGE_TOKEN)
            
#             i += 1
            
#         return new_ids

In [ ]:
# Reward Functions for RLVR

def format_reward(completions, **kwargs):
    """Reward for proper formatting with \\boxed{}"""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        # Check if response contains \\boxed{}
        if "\\boxed{" in response:
            scores.append(0.5)  # Small bonus for correct format
        else:
            scores.append(-0.5)  # Penalty for missing format
    return scores








def answer_reward(completions, answer, **kwargs):
    """
    Main reward: negative distance from correct answer
    Reward = -1 * abs(TRUE - PREDICTED) / abs(TRUE) if TRUE != 0
    Reward = -1 * abs(PREDICTED) if TRUE == 0
    Reward = 0 if exactly correct
    """
    scores = []
    
    for completion, true_ans in zip(completions, answer):
        response = completion[0]["content"]
        predicted = extract_boxed_answer(response)
        true_value = parse_true_answer(true_ans)
        
        if predicted is None or true_value is None:
            # Can't parse answer - give penalty
            scores.append(-2.0)
            continue
        
        # Calculate distance-based reward
        if abs(predicted - true_value) < 1e-6:
            # Exactly correct!
            reward = 10.0  # Bonus for correct answer
        else:
            # Calculate relative error
            if abs(true_value) > 1e-6:
                relative_error = abs(true_value - predicted) / abs(true_value)
            else:
                relative_error = abs(predicted)
            
            # Negative reward based on error, capped at -2
            reward = -1.0 * min(relative_error, 2.0)
        
        scores.append(reward)
    
    return scores

# Test the reward function
test_completions = [[{"content": "The answer is \\boxed{42}"}]]
test_answers = ["42"]
print(f"Test reward (correct): {answer_reward(test_completions, test_answers)}")

test_completions = [[{"content": "The answer is \\boxed{40}"}]]
print(f"Test reward (close): {answer_reward(test_completions, test_answers)}")

test_completions = [[{"content": "The answer is \\boxed{0}"}]]
print(f"Test reward (far): {answer_reward(test_completions, test_answers)}")

<h1 style="color:  #e323a9ff;">🗃️ Dataset</h1>

<h2 style="color: #e323a9ff;">🔁 Replicate Samples for More Steps</h2>

<div style="padding: 10px 12px; border-left: 6px solid #58d987ff; background: #F0FDF4; border-radius: 10px;">
Creates a Hugging Face <b>Dataset</b> and replicates entries so GRPO can run for many steps 🧩
</div>

In [ ]:
# Create the HuggingFace Dataset
from datasets import Dataset

# Replicate the 50 problems to have enough data for 100 steps
# With batch_size=1 and 100 steps, we need at least 100 samples
replicated_data = train_data * 2  # 50 problems * 2 = 100 samples

dataset = Dataset.from_list(replicated_data)

# Calculate prompt length for configuration
sample_prompt = tokenizer.apply_chat_template(
    train_data[0]["prompt"],
    tokenize=False,
    add_generation_prompt=True,
    reasoning_effort="low"
)
max_prompt_length = len(tokenizer(sample_prompt)["input_ids"]) + 10  # Add buffer

print(f"Dataset size: {len(dataset)}")
print(f"Max prompt length: {max_prompt_length}")
print(f"Sample formatted prompt:\n{sample_prompt[:500]}...")

<h1 style="color:  #e323a9ff;">🧪 GRPO</h1>

<h2 style="color: #e323a9ff;">⚙️ need to understand GRPO</h2>

<div style="padding: 10px 12px; border-left: 6px solid #3B82F6; background: #EFF6FF; border-radius: 10px;">
Sets sampling + optimization hyperparams.<br>
Watch <b>max_prompt_length</b> + <b>max_completion_length</b> to avoid truncation ✂️
</div>

In [ ]:
!uv pip install wandb 

In [ ]:
# import wandb
# wandb.login(key="YOUR_WANDB_API_KEY")

# run = wandb.init(
#     entity="barnoahmed666-none",
#     project="OSS GRPO ",
#     config={
#         "model_name": "unsloth/gpt-oss-20b",
#         "lora_rank": lora_rank,
#         "max_seq_length": max_seq_length,
#         "learning_rate": 5e-5,
#         "weight_decay": 0.001,
#         "warmup_ratio": 0.1,
#         "lr_scheduler_type": "linear",
#         "optim": "adamw_8bit",
#         "per_device_train_batch_size": 2,
#         "gradient_accumulation_steps": 1,
#         "num_generations": 2,
#         "max_steps": 100,
#         "temperature": 1.0,
#     },
# )


In [ ]:
# Configure GRPO Training
from trl import GRPOConfig, GRPOTrainer
import gc

# Cap completion length to something reasonable for speed
max_completion_length = 2048    #min(max_seq_length - max_prompt_length, 512)  # Cap at 512 tokens

training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 5e-5,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 1,
    num_generations = 2,  # Number of completions per prompt
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 100,  # 100 steps for testing
    save_steps = 50,
    report_to = None,
    output_dir = "outputs_grpo_test",
    #beta=0.01,
)

print(f"Max completion length: {max_completion_length}")
print("Training config ready!")

<h2 style="color: #e323a9ff;">🏗️ Build the Trainer</h2>

<div style="padding: 10px 12px; border-left: 6px solid #8B5CF6; background: #F5F3FF; border-radius: 10px;">
Wires together: <b>model</b> + <b>tokenizer</b> + <b>reward functions</b> + <b>dataset</b> 🧷
</div>

In [ ]:
# Initialize the GRPO Trainer
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward,    # Reward for using \boxed{}
        answer_reward,    # Main reward: distance-based correctness
    ],
    args = training_args,
    train_dataset = dataset,
)

print("Trainer initialized!")

<h1 style="color: #e323a9ff;">🚀 Training Run</h1>

<h2 style="color: #e323a9ff;">🏋️ GRPO Train</h2>

<div style="padding: 10px 12px; border-left: 6px solid #EF4444; background: #FEF2F2; border-radius: 10px;">
Heads up: generation dominates runtime ⏳<br>
If it’s too slow, reduce <b>max_steps</b>, <b>num_generations</b>, or <b>max_new_tokens</b>.
</div>

In [ ]:
# Start training - 100 steps
# Monitor the 'reward' column in the output table - it should increase over time
import time

print("Starting training... (This will take a while - generation is the slow part)")
start_time = time.time()
trainer.train()
end_time = time.time()

training_time = end_time - start_time
print(f"\n{'='*50}")
print(f"Training completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"Time per step: {training_time/100:.2f} seconds")
print(f"Estimated time for 1000 steps: {(training_time/100)*1000/60:.2f} minutes")

<h1 style="color: #e323a9ff;">🔎 Evaluation</h1>

<h2 style="color: #e323a9ff;">🧠 Quick Inference Smoke Test</h2>

<div style="padding: 10px 12px; border-left: 6px solid #10B981; background: #ECFDF5; border-radius: 10px;">
Generates on a known training prompt and prints the model output 🧪
</div>

In [ ]:
# Test inference after training
text = tokenizer.apply_chat_template(
    train_data[0]["prompt"],
    tokenize = False,
    add_generation_prompt = True,
    reasoning_effort = "low",
)

from transformers import TextStreamer

print("Testing trained model on first problem:")
print("="*50)
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0.7,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)
print(f"\nExpected answer: {train_data[0]['answer']}")

In [ ]:
# Save the LoRA adapters
model.save_pretrained("gpt_oss_20b_math_lora_test")
tokenizer.save_pretrained("gpt_oss_20b_math_lora_test")
print("LoRA adapters saved to 'gpt_oss_20b_math_lora_test'")

## Optional: Save merged model or push to Hub

Uncomment the options below as needed:
- **LoRA only**: Smallest size, requires base model to load
- **Merged 16bit**: Full model in fp16
- **MXFP4**: GPT-OSS native precision, good for VLLM

In [ ]:
# Optional: Merge and save in different formats

# Save merged model in MXFP4 (GPT-OSS native precision)
# model.save_pretrained_merged("gpt_oss_20b_math_mxfp4", tokenizer, save_method="mxfp4")

# Save merged model in 16bit
# model.save_pretrained_merged("gpt_oss_20b_math_16bit", tokenizer, save_method="merged_16bit")

# Push to Hugging Face Hub (uncomment and add your token)
# model.push_to_hub_merged("your-username/gpt-oss-20b-math", tokenizer, token="hf_...", save_method="lora")